In [ ]:
# app.py
import gradio as gr
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# ------- 1. Load your model -------
def load_model():
    model = YourColorizationModel()  # replace with your model class
    model.load_state_dict(torch.load("weights.pth", map_location="cpu"))
    model.eval()
    return model

model = load_model()

# ------- 2. Preprocess input -------
def preprocess(image: Image.Image):
    # Convert to grayscale (in case user uploads a color image)
    gray = image.convert("L")          # L = single channel grayscale
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)), # match your model's expected size
        transforms.ToTensor(),         # [0,1] range, shape: (1, H, W)
    ])
    
    tensor = transform(gray).unsqueeze(0)  # shape: (1, 1, H, W)
    return tensor

# ------- 3. Postprocess output -------
def postprocess(output_tensor, original_size):
    # output_tensor shape: (1, 3, H, W) — RGB output from your model
    output = output_tensor.squeeze(0)          # (3, H, W)
    output = output.clamp(0, 1)                # keep values in [0,1]
    output = output.permute(1, 2, 0).numpy()   # (H, W, 3)
    output = (output * 255).astype(np.uint8)   # back to 0-255
    
    result = Image.fromarray(output)
    result = result.resize(original_size, Image.LANCZOS)  # restore original size
    return result

# ------- 4. Main predict function -------
def colorize(image: Image.Image):
    original_size = image.size  # (W, H)
    
    input_tensor = preprocess(image)
    
    with torch.no_grad():
        output_tensor = model(input_tensor)
    
    colorized = postprocess(output_tensor, original_size)
    return colorized

# ------- 5. Gradio UI -------
demo = gr.Interface(
    fn=colorize,
    inputs=gr.Image(
        type="pil",
        label="Input — grayscale or color image"
    ),
    outputs=gr.Image(
        type="pil",
        label="Colorized Output"
    ),
    title="🎨 Image Colorization",
    description="Upload a grayscale (or color) image and the model will colorize it.",
    examples=[],          # optionally add: [["example1.jpg"], ["example2.jpg"]]
    flagging_mode="never" # disables the Flag button
)

demo.launch(share=True)